In [ ]:
import cudf as gd
import pandas as pd
import numpy as np
import tensorflow as tf
from collections import OrderedDict
import time
from tqdm import tqdm
import pickle
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore")
print('cudf version:',gd.__version__)
print('tensorflow version:',tf.__version__)

### Please download data from https://www.kaggle.com/c/santander-customer-transaction-prediction/data

In [ ]:
PATH = '../input'

# ETL

### Read csv

In [ ]:
%%time
cols = ['ID_code', 'target'] + ['var_%d'%i for i in range(200)]
dtypes = ['int32', 'int32'] + ['float32' for i in range(200)]
train_gd = gd.read_csv('%s/train.csv'%PATH,names=cols,dtype=dtypes,skiprows=1)

cols = ['ID_code', 'target'] + ['var_%d'%i for i in range(200)]
dtypes = ['int32', 'int32'] + ['float32' for i in range(200)]
test_gd = gd.read_csv('%s/test.csv'%PATH,names=cols,dtype=dtypes,skiprows=1)

In [ ]:
train_gd.head().to_pandas()

### create new features & normalize


In [ ]:
%%time
for i in tqdm(range(200)):
    col = 'var_%d'%i
    new_col = 'new_%s'%col
    count_col = 'count_%s'%col
    
    df = train_gd.groupby(col).agg({col:'count'})
    df = df.reset_index()
    train_gd = train_gd.merge(df,on=col,how='left')
    test_gd = test_gd.merge(df,on=col,how='left')
    
    # feature values with count==1 have a lot of noise
    # we can replace these values with mean value of the column
    train_gd[new_col] = train_gd[col] * (train_gd[count_col]>1)
    mean = train_gd[new_col].mean()
    std = train_gd[new_col].std()
    train_gd['mean'] = mean
    train_gd[new_col] = train_gd[new_col] + train_gd['mean']*(train_gd[count_col]==1)
    train_gd[new_col] = (train_gd[new_col]-mean)/std
    train_gd[col] = (train_gd[col]-mean)/std
    
    test_gd[new_col] = test_gd[col] * (test_gd[count_col]>1)
    test_gd['mean'] = mean
    test_gd[new_col] = test_gd[new_col] + test_gd['mean']*(test_gd[count_col]==1)
    test_gd[new_col] = (test_gd[new_col]-mean)/std
    test_gd[col] = (test_gd[col]-mean)/std
    

In [ ]:
feas = []
for i in range(200):
    feas.append('var_%d'%i)
    feas.append('new_var_%d'%i)
X = train_gd[feas].to_pandas()
X.head()

In [ ]:
X = X.values
y = train_gd['target'].to_pandas().values
Xt = test_gd[feas].to_pandas().values
print(X.shape,y.shape,Xt.shape)

In [ ]:
print(X.shape,y.shape,Xt.shape)
B = X.shape[0]
X = np.reshape(X,[B,200,2])
Xt = np.reshape(Xt,[B,200,2])
print(X.shape,y.shape,Xt.shape)

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Define TF model

In [ ]:
class groupNN:
    def __init__(self,**params):
        self.params = params
        
    def fit(self,X,y):
        self.X = X
        self.y = y
        tf.reset_default_graph()
        # build a tf computing graph
        logit = self._build()
        label = tf.placeholder(dtype=tf.int32,shape=[None]) # B,classes
        losst = self.get_loss(logit,label)
        opt_op = self.get_opt(losst)
        
        with tf.Session() as sess:
            sess.run(tf.global_variables_initializer())
            sess.run(tf.local_variables_initializer())
            #self.load(sess)
            i = 1
            ave_loss = []
            for c,(Xb,yb,end_epoch) in enumerate(self._batch_gen(shuffle=True)):
                loss,_ = sess.run([losst,opt_op],feed_dict={self.inputs:Xb, label:yb})
                ave_loss.append(loss)
                if end_epoch:
                    print("Epoch %d train loss %.4f"%(i,np.mean(ave_loss)))
                    i += 1
                    ave_loss = []
            self.save(sess)
    
    def _build(self):
        # build the computing graph      
        netname = 'groupNN'
        self.inputs = tf.placeholder(dtype=tf.float32,shape=[None,200,2])
        B = tf.shape(self.inputs)[0]
        H = self.params.get('hidden_units', 16)
        with tf.variable_scope(netname):
            net = self.inputs
            net = tf.contrib.layers.fully_connected(self.inputs,H)
            net = tf.reshape(net,[B,200*H])
            net = tf.contrib.layers.fully_connected(net,1,activation_fn=None)
            return tf.squeeze(net)
        
    def predict(self,X):
        print('predict')
        self.X = X 
        self.y = None
        tf.reset_default_graph()
        self.params['epochs'] = 1
        # build a tf computing graph
        logit = self._build()
        logit = tf.nn.sigmoid(logit)
        preds = []
        #print('here')
        with tf.Session() as sess:
            sess.run(tf.global_variables_initializer())
            sess.run(tf.local_variables_initializer())
            self.load(sess)
            for c,(Xb,_,end_epoch) in enumerate(self._batch_gen(shuffle=False)):
                pred = sess.run(logit,feed_dict={self.inputs:Xb})
                preds.append(pred)
        preds = np.concatenate(preds)
        return preds
    
    def get_opt(self,loss):
        learning_rate = self.params.get('learning_rate', 0.001)
        opt = tf.train.AdamOptimizer(learning_rate=learning_rate)
        return opt.minimize(loss)
        
    def get_loss(self, logit, label):
        # build the loss tensor
        label = tf.cast(label,tf.float32)
        loss = tf.nn.sigmoid_cross_entropy_with_logits(logits=logit,labels=label)
        return tf.reduce_mean(loss)
    
    def save(self, sess):
        varss = tf.trainable_variables()
        weights = {} # var.name => var.value: a numpy array
        for var in varss:
            val = sess.run(var)
            weights[var.name] = val
        pickle.dump(weights,open('weight.p','wb'))

    def load(self, sess, path = 'weight.p'):
        weights = pickle.load(open(path,'rb'))
        varss = tf.trainable_variables()
        for var in varss:
            value = weights[var.name]
            assign_op = var.assign(value)
            sess.run(assign_op)
    
    def _batch_gen(self, shuffle=True):
        X,y = self.X, self.y
        B = self.params.get('batch_size', 1024)
        epochs = self.params.get('epochs', 10)
        ids = [i for i in range(len(X))]
        batches = len(X)//B + 1
        #print(epochs,batches)
        for epoch in range(epochs):
            if shuffle:
                random.shuffle(ids)
            for i in range(batches):                
                idx = ids[i*B:(i+1)*B]
                if y is not None:
                    yield X[idx],y[idx],i==batches-1
                else:
                    yield X[idx],None,i==batches-1
            if (i+1)*B < len(X):
                idx = ids[(i+1)*B:len(X)]
                yield X[idx]

In [ ]:
nn = groupNN(hidden_units=16,learning_rate=0.01,epochs=20)

In [ ]:
nn.fit(X_train,y_train)

In [ ]:
yp = nn.predict(X_valid)
print('Validation AUC',roc_auc_score(y_valid,yp))

In [ ]:
yp = nn.predict(Xt)

In [ ]:
submission = pd.DataFrame({'ID_code':test_gd['ID_code'].to_pandas().values,
                   'target':yp})
submission.to_csv('submission.csv',index=False)